# Olist E-Commerce Business Intelligence

## Phase 5 — Business Intelligence Dashboard

### Objective

This notebook transforms the cleaned data into business intelligence by developing key performance indicators (KPIs) and answering strategic business questions.

Topics covered:

- Executive KPIs
- Sales Performance
- Customer Insights
- Product Insights
- Seller Performance
- Logistics Performance
- Customer Satisfaction
- Strategic Recommendations

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
PROJECT_ROOT = Path().resolve().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

In [4]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

customers = pd.read_csv(PROCESSED_PATH / "customers_clean.csv")

geolocation = pd.read_csv(PROCESSED_PATH / "geolocation_clean.csv")

order_items = pd.read_csv(PROCESSED_PATH / "order_items_clean.csv")

payments = pd.read_csv(PROCESSED_PATH / "payments_clean.csv")

reviews = pd.read_csv(PROCESSED_PATH / "reviews_clean.csv")

orders = pd.read_csv(
    PROCESSED_PATH / "orders_clean.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

products = pd.read_csv(PROCESSED_PATH / "products_clean.csv")

sellers = pd.read_csv(PROCESSED_PATH / "sellers_clean.csv")

translation = pd.read_csv(PROCESSED_PATH / "translation_clean.csv")

In [6]:
print("Total Customers :", customers.shape[0])

print("Total Orders :", orders.shape[0])

print("Total Products :", products.shape[0])

print("Total Sellers :", sellers.shape[0])

print("Total Payments :", payments.shape[0])

print("Total Reviews :", reviews.shape[0])

Total Customers : 99441
Total Orders : 99441
Total Products : 32951
Total Sellers : 3095
Total Payments : 103886
Total Reviews : 99224


In [8]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [9]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [10]:
orders[["order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"]].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


In [11]:
average_delivery = orders["delivery_days"].mean()

In [12]:
total_customers = customers["customer_id"].nunique()

total_orders = orders["order_id"].nunique()

total_products = products["product_id"].nunique()

total_sellers = sellers["seller_id"].nunique()

total_revenue = payments["payment_value"].sum()

average_order_value = payments["payment_value"].mean()

average_review = reviews["review_score"].mean()

average_delivery = orders["delivery_days"].mean()

In [13]:
print("="*60)
print("Executive Dashboard")
print("="*60)

print(f"Customers           : {total_customers:,}")
print(f"Orders              : {total_orders:,}")
print(f"Products            : {total_products:,}")
print(f"Sellers             : {total_sellers:,}")
print(f"Revenue             : R$ {total_revenue:,.2f}")
print(f"Average Order Value : R$ {average_order_value:.2f}")
print(f"Average Rating      : {average_review:.2f}")
print(f"Average Delivery    : {average_delivery:.2f} Days")

Executive Dashboard
Customers           : 99,441
Orders              : 99,441
Products            : 32,951
Sellers             : 3,095
Revenue             : R$ 16,008,872.12
Average Order Value : R$ 154.10
Average Rating      : 4.09
Average Delivery    : 12.09 Days


Business Question 1
Which states generate the highest revenue?

In [14]:
sales = (
    payments
    .merge(orders,on="order_id")
    .merge(customers,on="customer_id")
)

In [15]:
state_revenue = (
    sales
    .groupby("customer_state")["payment_value"]
    .sum()
    .sort_values(ascending=False)
)

Business Question 2
Which product categories generate the highest revenue?

In [16]:
product_sales = (
    order_items
    .merge(products,on="product_id")
    .merge(translation,on="product_category_name",how="left")
    .merge(payments,on="order_id")
)

In [17]:
category_revenue = (
    product_sales
    .groupby("product_category_name_english")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

Business Question 3
Which sellers generate the highest revenue?

In [18]:
seller_sales = (
    order_items
    .merge(payments,on="order_id")
)

In [19]:
top_seller = (
    seller_sales
    .groupby("seller_id")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

Business Question 4
Does delivery speed affect ratings?

In [20]:
review_orders = (
    reviews
    .merge(orders,on="order_id")
)

Business Question 5
Monthly Revenue Growth

In [21]:
monthly = (
    payments
    .merge(orders,on="order_id")
)

monthly["month"] = (
    monthly["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_growth = (
    monthly
    .groupby("month")["payment_value"]
    .sum()
)

# Executive Business Report

## Revenue

- Revenue is concentrated in a limited number of product categories and geographic regions.
- Credit card transactions account for the largest share of sales revenue.

## Customers

- Customer satisfaction is generally high, with most reviews receiving positive ratings.
- Delivery performance appears to influence review scores.

## Products

- A small number of categories generate the majority of demand.
- Product portfolio optimization should prioritize these high-performing categories.

## Sellers

- Seller performance is uneven, with a limited number of sellers contributing a significant share of marketplace activity.
- High-performing sellers can serve as benchmarks for operational best practices.

## Logistics

- Delivery times are acceptable for most orders, though opportunities exist to reduce delays for a subset of customers.
- Improving logistics efficiency may lead to higher customer satisfaction and repeat purchases.

## Strategic Recommendations

1. Invest marketing resources in high-revenue states.
2. Expand inventory for top-performing product categories.
3. Strengthen partnerships with high-performing sellers.
4. Improve logistics in regions with longer delivery times.
5. Encourage customer reviews to gain richer feedback for sentiment analysis.